In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('Titanic-Dataset.csv')
print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [4]:
df.drop(['PassengerId','Name','Ticket','Cabin'] , axis =1 , inplace=True)
df.info()

# Let us Now Start our ML Workflow
y = df.iloc[:,0]
X = df.iloc[:,1:]

print(X.shape)
print(y.shape)

df.head()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    str    
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 60.9 KB
(891, 7)
(891,)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

numerical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

clf = ColumnTransformer(
    transformers=[
        ('num', numerical_pipe, ['Age', 'Fare']),
        ('cat', categorical_pipe, ['Embarked']),
        ('sex', OneHotEncoder(drop='if_binary'), ['Sex'])
    ],
    remainder='passthrough'
)

pipe = Pipeline([
    ('clf', clf),
    ('model', LogisticRegression(max_iter=1000))
])

param_grid = [

    # L2
    {
        'model__l1_ratio': [0],
        'model__solver': ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga'],
        'model__C': [0.01, 0.1, 1, 10, 100]
    },

    # L1
    {
        'model__l1_ratio': [1],
        'model__solver': ['liblinear', 'saga'],
        'model__C': [0.01, 0.1, 1, 10, 100]
    },

    # Elastic Net
    {
        'model__solver': ['saga'],
        'model__C': [0.01, 0.1, 1, 10, 100],
        'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
    }
]

grid = GridSearchCV(estimator = pipe ,
                    param_grid = param_grid,
                    cv = 10,
                    scoring='accuracy',
                    n_jobs = -1
                   )

grid.fit(X,y)



print(f'Best Scores -> {grid.best_score_}')
print(f'Best Parameters -> {grid.best_params_}')


Best Scores -> 0.7968913857677903
Best Parameters -> {'model__C': 0.1, 'model__l1_ratio': 0.1, 'model__solver': 'saga'}
